In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"Using device: {device}")


In [ ]:
from pathlib import Path

text = Path('data/tiny-shakespeare.txt').read_text()

In [ ]:
print(text[0:1000])

In [ ]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [ ]:
tokenizer = CharTokenizer.train_from_text(text)

In [ ]:
print(tokenizer.encode("Hello world"))

In [ ]:
print(tokenizer.decode(tokenizer.encode("Hello world")))

In [ ]:
tokenizer.vocabulary_size()

In [ ]:
import pprint
pp = pprint.PrettyPrinter(depth=4)

In [ ]:
pp.pprint(tokenizer.char_for_token_id)

In [ ]:
pp.pprint(tokenizer.token_id_for_char)

# Exercise: Implementing a Data Loader

In [ ]:
from pathlib import Path

text = Path("data/tiny-shakespeare.txt").read_text()

In [ ]:
print(text[0:1000])

In [ ]:
import torch


class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [ ]:
tokenizer = CharTokenizer.train_from_text(text)

In [ ]:
print(tokenizer.encode("Hello world"))
print(tokenizer.decode(tokenizer.encode("Hello world")))

In [ ]:
print(f"Vocabulary size: {tokenizer.vocabulary_size()}")

In [ ]:
# Step 1 - Define the `TokenIdsDataset` Class

from torch.utils.data import Dataset

class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # If every position can be a start of an item, the final
        # valid start is len(data) - block_size - 1.
        return len(self.data) - self.block_size
        # of the dataset

    def __getitem__(self, pos):
        if pos < 0 or pos >= len(self):
            raise IndexError("Dataset index out of range")

        item = self.data[pos : pos + self.block_size]
        target = self.data[pos + 1 : pos + self.block_size + 1]

        return item, target


In [ ]:
# Step 2 - Tokenize the Text

tokenized_text = tokenizer.encode(text)
dataset = TokenIdsDataset(tokenized_text, block_size=64)

In [ ]:
# Step 3 - Retrieve the First Item from the Dataset

x, y = dataset[0]
print(tokenizer.decode(x))

In [ ]:
from torch.utils.data import DataLoader, RandomSampler

# RandomSampler allows to read random items from a datasset
sampler = RandomSampler(dataset, replacement=True)
# Dataloader will laod two random samplers using the sampler
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

In [ ]:
# Step 4 - Use a DataLoader

batch_x, batch_y = next(iter(dataloader))

In [ ]:
print(tokenizer.decode(batch_x[0]))

In [ ]:
print(tokenizer.decode(batch_y[0]))